In [ ]:
import csv
import time
import datetime
from sklearn import metrics

from nltk import tag
from nltk.tag import map_tag
from nltk import word_tokenize
from nltk.tag.mapping import map_tag

from nltk.tag.tnt import TnT
tagger_name = "TnT"

from nltk.corpus import brown as corpus
corpus_name = "Brown"


In [ ]:
# TRAINING

# Training set
# First 105 files of the Brown corpus

training_sentences = corpus.tagged_sents(corpus.fileids()[0:105])

tagger = TnT()

start = time.time()
tagger.train(training_sentences)
end = time.time()
processing_time = end - start

print('Training has lasted: ', processing_time, 'sec' )


In [ ]:
# TESTING 1:
# Testing set
# 50 files, from 106 to 155 of the Brown Corpus

# Text del corpus de test amb etiquetes originals en una llista de tuples (token, etiqueta)

testing_sentences = corpus.tagged_sents(corpus.fileids()[105:156])

print("Test set Gold labels")
print(testing_sentences[0][0:3],testing_sentences[1][0:3])

# Text del corpus en una llista de paraules

bare_sentences = [([word for word,_ in sentence]) for sentence in testing_sentences]
   # ALTERNATIVA: bare_sentences = corpus.sents(text_id)
    
print(" ")
print("Words:")
print(bare_sentences[0][0:3], bare_sentences[1][0:3])
    

In [ ]:
# TESTING 2: 
# Predicció d'etiquetes del corpus de test

start = time.time()
predicted_sentences = [tagger.tag([word for word,_ in sentence]) for sentence in testing_sentences]
end = time.time()
processing_time = end - start

print(" ")
print("Predicted labels (tagger default's tagset):")
print(predicted_sentences[0][0:3], predicted_sentences[1][0:3])
print('Tagging has lasted: ', processing_time, 'sec' )

# Canvi de tagset: Universal

for i in range(len(predicted_sentences)):
    predicted_sentences[i] = [(word, tag, map_tag('en-ptb', 'universal', tag)) for word, tag in predicted_sentences[i]]

print(" ")
print("Predicted labels (Tagset 'Universal'):")
print(predicted_sentences[0][0:3], predicted_sentences[1][0:3])


In [ ]:
# EVALUATION 1
# Gold and predicted labels of all sentences joined in one single list, respectivament.

golds = [tag for sentence in testing_sentences for _,tag in sentence]
predicted_labels = [tag for sentence in predicted_sentences for _,tag,_ in sentence]

print("Gold labels:")
print(golds[0:3])
print(" ")
print("Predicted labels:")
print(predicted_labels[0:3])



In [ ]:
# EVALUATION 2
# Metrics

for preds,tagger in [(predicted_labels,tagger)]:

    accuracy = metrics.accuracy_score(golds,preds)
    precision = metrics.precision_score(golds,preds,average='weighted')
    recall = metrics.recall_score(golds,preds,average='weighted')
    f1_score = metrics.f1_score(golds,preds,average='weighted')
        
    print("Metrics for",tagger)   
    print(" Accuracy :", accuracy)
    print(" Precision:", precision)
    print(" Recall   :", recall)
    print(" F1-Score :", f1_score)


In [ ]:
# EXPORTACIÓ DE les primeres 15 línies del corpus de test.

now = datetime.datetime.now().strftime("%y%m%d-%H%M")

all_labels = testing_sentences[0:15].copy()

with open(now + '-' + tagger_name + '-' + corpus_name + '-.csv', mode='w') as dades:
    dades_writer = csv.writer(dades, delimiter=',')
    for i in range(len(testing_sentences[0:15])):
        for j in range (len(testing_sentences[0:15][i])):
            all_labels[i][j] = testing_sentences[0:15][i][j] + predicted_sentences[0:15][i][j]
            dades_writer.writerow(all_labels[i][j])

print(" ")
print("Gold vs predicted labels (Tagset 'Universal'):")
print(all_labels[0][0:3], all_labels[1][0:3])

